# Transition maps by era

State-to-state transitions from the BEAST jump history, visualized as arrows on a map.
One map per era (cryptic, detection, post-order) plus total.

In [1]:
import re
import warnings
from collections import defaultdict
from datetime import datetime, timedelta
from pathlib import Path

import geopandas as gpd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd

warnings.filterwarnings('ignore', category=FutureWarning)

plt.rcParams.update({
    'font.family': 'Arial',
    'font.size': 9,
    'figure.dpi': 300,
    'savefig.dpi': 300,
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
})

PROJ_ROOT = Path('../..').resolve()
FINAL_DIR = PROJ_ROOT / 'paper' / 'final'
BEAST_DIR = FINAL_DIR / 'analysis/beast'
FIG_DIR = FINAL_DIR / 'figures'

_CORR2E_DIR = BEAST_DIR / 'phylogeography/correlateSet2e'
_PREFIX = 'timeinhomogeneous_correlateSet2enforced_weeklyEpochs_3era_3rateScalar'
JUMP_HISTORY_LOG = _CORR2E_DIR / f'{_PREFIX}.jumpHistory.log'

SHP_PATH = (FINAL_DIR.parent.parent / 'analysis' /
            'H5N1-State-and-County-Simulations-and-Analysis_edit' /
            'data' / 'gadm41_USA_shp' / 'gadm41_USA_1.shp')

MRTP = 2025.8219
TX_DETECT = datetime(2024, 3, 25)
FED_ORDER = datetime(2024, 4, 29)

STATE_ORDER = ['CA', 'CO', 'IA', 'ID', 'KS', 'MI', 'MN', 'NC',
               'NE', 'NM', 'OH', 'OK', 'SD', 'TX', 'UT', 'WY']

print('Setup complete.')

Setup complete.


In [2]:
# Load jump history
_JUMP_PATTERN = re.compile(r'\{1,([\d.]+),(\w+),(\w+)\}')

def decimal_to_date(dec_year):
    year = int(dec_year)
    remainder = dec_year - year
    return datetime(year, 1, 1) + timedelta(days=remainder * 365.25)

def assign_era(dt):
    if dt < TX_DETECT:
        return 'Pre-detection'
    elif dt < FED_ORDER:
        return 'Pre-response'
    else:
        return 'Post-response'

print('Loading jump history...')
histories = []
with open(JUMP_HISTORY_LOG) as f:
    for line in f:
        if line.startswith('#') or line.startswith('state'):
            continue
        parts = line.strip().split('\t')
        if len(parts) < 3:
            continue
        histories.append(parts[2])

n_burnin = len(histories) // 10
histories = histories[n_burnin:]
print(f'{len(histories)} post-burnin samples')

samples = []
for hist_str in histories:
    matches = _JUMP_PATTERN.findall(hist_str)
    jumps = []
    for h_str, src, dest in matches:
        cal_date = decimal_to_date(MRTP - float(h_str))
        jumps.append((cal_date, src, dest))
    samples.append(jumps)

print(f'Loaded {len(samples)} samples.')

Loading jump history...
90001 post-burnin samples
Loaded 90001 samples.


In [3]:
# Compute posterior median and mean transition matrices by era
s2i = {s: i for i, s in enumerate(STATE_ORDER)}
n_states = len(STATE_ORDER)
n_samples = len(samples)

era_names = ['All', 'Pre-detection', 'Pre-response', 'Post-response']
matrices = {era: np.zeros((n_samples, n_states, n_states)) for era in era_names}

for si, jumps in enumerate(samples):
    for cal_date, src, dest in jumps:
        if src in s2i and dest in s2i and src != dest:
            ri, ci = s2i[src], s2i[dest]
            matrices['All'][si, ri, ci] += 1
            era = assign_era(cal_date)
            matrices[era][si, ri, ci] += 1

median_matrices = {era: np.median(matrices[era], axis=0) for era in era_names}
mean_matrices = {era: np.mean(matrices[era], axis=0) for era in era_names}

for era in era_names:
    per_sample_totals = np.sum(matrices[era], axis=(1, 2))
    print(f'{era}: median {round(np.median(per_sample_totals))} transitions')
    print(f'{era}: mean {round(np.mean(per_sample_totals))} transitions')

All: median 66 transitions
All: mean 66 transitions
Pre-detection: median 29 transitions
Pre-detection: mean 29 transitions
Pre-response: median 21 transitions
Pre-response: mean 21 transitions
Post-response: median 16 transitions
Post-response: mean 16 transitions


In [4]:
# Load US states shapefile and compute centroids
us = gpd.read_file(SHP_PATH)
us['abbr'] = us['HASC_1'].str.replace('US.', '', regex=False)

# Filter to CONUS
exclude = {'AK', 'HI', 'DC', 'AS', 'GU', 'MP', 'PR', 'VI'}
conus = us[~us['abbr'].isin(exclude)].copy()
conus = conus.to_crs('EPSG:2163')  # US National Atlas Equal Area

# Compute centroids
centroids = {}
for _, row in conus.iterrows():
    c = row.geometry.centroid
    centroids[row['abbr']] = (c.x, c.y)

# Check which study states are in the shapefile
for s in STATE_ORDER:
    if s not in centroids:
        print(f'WARNING: {s} not found in shapefile')

print(f'CONUS states: {len(conus)}, study state centroids: {sum(s in centroids for s in STATE_ORDER)}')

CONUS states: 48, study state centroids: 16


In [5]:
# ── Arrow polygon utilities (style 1: single filled polygon) ──

def offset_curve(pts, offset):
    n = len(pts)
    normals = np.zeros_like(pts)
    for i in range(n):
        if i == 0:
            d = pts[1] - pts[0]
        elif i == n - 1:
            d = pts[-1] - pts[-2]
        else:
            d = pts[i + 1] - pts[i - 1]
        length = np.linalg.norm(d)
        if length == 0:
            normals[i] = [0, 0]
        else:
            d = d / length
            normals[i] = np.array([-d[1], d[0]])
    return pts + normals * offset


def quadratic_bezier(p0, p1, ctrl, n=200):
    t = np.linspace(0, 1, n)[:, None]
    return (1 - t)**2 * p0 + 2 * (1 - t) * t * ctrl + t**2 * p1


def build_arrow_polygon(src, dst, weight, w_min, w_max,
                        shrink=70000, arc_frac=0.20,
                        head_len_base=40000, head_w_ratio=0.65,
                        shaft_w_min=8000, shaft_w_max=25000):
    s = np.array(src, dtype=float)
    d = np.array(dst, dtype=float)
    vec = d - s
    L = np.linalg.norm(vec)
    if L < 2 * shrink:
        return None
    uv = vec / L
    s_sh = s + uv * shrink
    d_sh = d - uv * shrink

    t_w = (weight - w_min) / (w_max - w_min) if w_max != w_min else 0.5
    sw = shaft_w_min + t_w * (shaft_w_max - shaft_w_min)
    head_len = head_len_base * (0.7 + 0.5 * t_w)
    hw = max(head_len * head_w_ratio, sw * 2.0)

    mid = (s_sh + d_sh) / 2
    perp = np.array([-uv[1], uv[0]])
    ctrl = mid + perp * arc_frac * np.linalg.norm(d_sh - s_sh)

    full_pts = quadratic_bezier(s_sh, d_sh, ctrl, n=200)
    diffs = np.diff(full_pts, axis=0)
    seg_lens = np.sqrt((diffs**2).sum(axis=1))
    cum_len = np.concatenate([[0], np.cumsum(seg_lens)])
    total_len = cum_len[-1]

    shaft_end_len = max(total_len * 0.5, total_len - head_len)
    idx_cut = min(np.searchsorted(cum_len, shaft_end_len), len(full_pts) - 2)

    shaft_pts = full_pts[:idx_cut + 1]
    head_base = full_pts[idx_cut]
    tip = full_pts[-1]

    left_shaft = offset_curve(shaft_pts, sw / 2)
    right_shaft = offset_curve(shaft_pts, -sw / 2)

    if idx_cut < len(full_pts) - 1:
        head_dir = full_pts[idx_cut + 1] - full_pts[idx_cut]
    else:
        head_dir = full_pts[-1] - full_pts[-2]
    head_dir = head_dir / np.linalg.norm(head_dir)
    head_perp = np.array([-head_dir[1], head_dir[0]])

    hb_left = head_base + head_perp * (hw / 2)
    hb_right = head_base - head_perp * (hw / 2)

    poly = np.vstack([
        left_shaft, hb_left[None, :], tip[None, :],
        hb_right[None, :], right_shaft[::-1],
    ])
    return poly


print('Arrow utilities defined.')

Arrow utilities defined.


In [6]:
# ── Combined 2×4 figure: heatmaps (top) + maps (bottom) ──
import matplotlib.patheffects as pe
from matplotlib.patches import Polygon as MplPolygon

# Palette
P = {
    'bg': '#ffffff', 'land': '#ededed', 'study': '#d6dfe8',
    'border': '#d0d0d0', 'label': '#2c3e50', 'arrow': '#34495e',
}

era_order = ['All', 'Pre-detection', 'Pre-response', 'Post-response']

# Build edges per era
def get_edges(mat):
    edges = []
    for i, src in enumerate(STATE_ORDER):
        for j, dest in enumerate(STATE_ORDER):
            if i != j and mat[i, j] > 0:
                edges.append((src, dest, mat[i, j]))
    return edges

all_edges = {era: get_edges(median_matrices[era]) for era in era_order}

# Median total transitions per era (for titles)
era_n = {}
for era in era_order:
    per_sample = np.sum(matrices[era], axis=(1, 2))
    era_n[era] = round(np.median(per_sample))

era_titles = [f'All eras ({era_n["All"]} transitions)',
              f'Pre-detection ({era_n["Pre-detection"]} transitions)',
              f'Pre-response ({era_n["Pre-response"]} transitions)',
              f'Post-response ({era_n["Post-response"]} transitions)']

# Global max/min weight from the All panel for consistent arrow scaling
global_w_max = max(e[2] for e in all_edges['All'])
global_w_min = min(e[2] for e in all_edges['All'])
global_heatmap_max = max(e[2] for edges in all_edges.values() for e in edges)
print(f'Arrow scaling: global w_min={global_w_min}, w_max={global_w_max}')

fig, axes = plt.subplots(2, 4, figsize=(20, 10),
                         gridspec_kw={'height_ratios': [1, 1.15], 'hspace': -0.05, 'wspace': 0.05})
fig.patch.set_facecolor('white')

for col, era in enumerate(era_order):
    mat = median_matrices[era]
    edges = all_edges[era]

    # ── Top row: heatmap ──
    ax_h = axes[0, col]
    im = ax_h.imshow(mat, cmap='Blues', vmin=0, vmax=global_heatmap_max,
                     aspect='equal', interpolation='nearest')
    
    ax_h.set_xticks(range(n_states))
    ax_h.set_xticklabels(STATE_ORDER, fontsize=7, rotation=90)
    ax_h.set_yticks(range(n_states))
    if col == 0:
        ax_h.set_yticklabels(STATE_ORDER, fontsize=7)
        ax_h.set_ylabel('Origin', fontsize=8)
    else:
        ax_h.set_yticklabels([])
    ax_h.tick_params(length=2, pad=2)
    ax_h.set_xlabel('Destination', fontsize=8)
    ax_h.set_title(era_titles[col], fontsize=10, fontweight='bold', pad=6)

    # ── Bottom row: map ──
    ax_m = axes[1, col]
    ax_m.set_facecolor(P['bg'])
    
    conus.plot(ax=ax_m, facecolor=P['land'], edgecolor=P['border'], linewidth=0.3)
    study_gdf = conus[conus['abbr'].isin(STATE_ORDER)]
    study_gdf.plot(ax=ax_m, facecolor=P['study'], edgecolor=P['border'], linewidth=0.3)
    
    if edges:
        for src, dest, w in sorted(edges, key=lambda e: e[2]):
            if src not in centroids or dest not in centroids:
                continue
            poly = build_arrow_polygon(
                centroids[src], centroids[dest], w, global_w_min, global_w_max)
            if poly is None:
                continue
            patch = MplPolygon(poly, closed=True, facecolor=P['arrow'],
                              edgecolor='none', alpha=1.0, zorder=3)
            ax_m.add_patch(patch)
    
    halo = [pe.withStroke(linewidth=2.0, foreground='white')]
    for s in STATE_ORDER:
        if s in centroids:
            x, y = centroids[s]
            ax_m.text(x, y, s, ha='center', va='center', fontsize=5.5,
                     fontweight='bold', color=P['label'], zorder=5,
                     fontfamily='Arial', path_effects=halo)
    
    ax_m.set_xlim(conus.total_bounds[0] - 80000, conus.total_bounds[2] + 80000)
    ax_m.set_ylim(conus.total_bounds[1] - 80000, conus.total_bounds[3] + 80000)
    ax_m.set_aspect('equal')
    ax_m.axis('off')

# Colorbar
cbar_ax = fig.add_axes([0.92, 0.55, 0.008, 0.30])
cbar = fig.colorbar(im, cax=cbar_ax)
cbar.set_label('Median transitions', fontsize=10)
cbar.ax.tick_params(labelsize=9)

# Panel labels (lowercase)
for i, label in enumerate(['a', 'b', 'c', 'd']):
    axes[0, i].text(-0.05, 1.05, label, transform=axes[0, i].transAxes,
                    fontsize=12, fontweight='bold', va='top')
for i, label in enumerate(['e', 'f', 'g', 'h']):
    axes[1, i].text(0.0, 1.02, label, transform=axes[1, i].transAxes,
                    fontsize=12, fontweight='bold', va='top')

# fig.savefig(FIG_DIR / 'S_TransitionMaps_2x4.median.png', dpi=300,
#             bbox_inches='tight', facecolor='white')
# fig.savefig(FIG_DIR / 'S_TransitionMaps_2x4.median.pdf', dpi=300,
#             bbox_inches='tight', facecolor='white')
plt.show()
# print('Saved 2x4 figure.')

Arrow scaling: global w_min=1.0, w_max=13.0


/var/folders/ls/3swb_7515hv8jb0mjxshss480000gp/T/ipykernel_98066/262381559.py:117: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [7]:
# ── Combined 2×4 figure: heatmaps (top) + maps (bottom) ──
import matplotlib.patheffects as pe
from matplotlib.patches import Polygon as MplPolygon

# Palette
P = {
    'bg': '#ffffff', 'land': '#ededed', 'study': '#d6dfe8',
    'border': '#d0d0d0', 'label': '#2c3e50', 'arrow': '#34495e',
}

era_order = ['All', 'Pre-detection', 'Pre-response', 'Post-response']

# Build edges per era
def get_edges(mat):
    edges = []
    for i, src in enumerate(STATE_ORDER):
        for j, dest in enumerate(STATE_ORDER):
            if i != j and round(mat[i, j]) > 0:
                edges.append((src, dest, round(mat[i, j])))
    return edges

all_edges = {era: get_edges(mean_matrices[era]) for era in era_order} # these are rounded to nearest integer to only show whole number of transitions on the map

# Mean total transitions per era (for titles)
era_n = {}
for era in era_order:
    per_sample = np.sum(matrices[era], axis=(1, 2))
    era_n[era] = round(np.mean(per_sample))

era_titles = [f'All eras ({era_n["All"]} transitions)',
              f'Pre-detection ({era_n["Pre-detection"]} transitions)',
              f'Pre-response ({era_n["Pre-response"]} transitions)',
              f'Post-response ({era_n["Post-response"]} transitions)']

# Global max/min weight from the All panel for consistent arrow scaling
global_w_max = max(e[2] for e in all_edges['All'])
global_w_min = min(e[2] for e in all_edges['All'])
global_heatmap_max = max(e[2] for edges in all_edges.values() for e in edges)
print(f'Arrow scaling: global w_min={global_w_min}, w_max={global_w_max}')

fig, axes = plt.subplots(2, 4, figsize=(20, 10),
                         gridspec_kw={'height_ratios': [1, 1.15], 'hspace': -0.05, 'wspace': 0.05})
fig.patch.set_facecolor('white')

for col, era in enumerate(era_order):
    mat = mean_matrices[era]
    edges = all_edges[era]

    # ── Top row: heatmap ──
    ax_h = axes[0, col]
    # not using the rounded matrix here, so that the heatmap shows the mean values (which can be fractional) rather than just whole numbers
    im = ax_h.imshow(mat, cmap='Blues', vmin=0, vmax=global_heatmap_max, 
                     aspect='equal', interpolation='nearest')
    
    ax_h.set_xticks(range(n_states))
    ax_h.set_xticklabels(STATE_ORDER, fontsize=7, rotation=90)
    ax_h.set_yticks(range(n_states))
    if col == 0:
        ax_h.set_yticklabels(STATE_ORDER, fontsize=7)
        ax_h.set_ylabel('Origin', fontsize=8)
    else:
        ax_h.set_yticklabels([])
    ax_h.tick_params(length=2, pad=2)
    ax_h.set_xlabel('Destination', fontsize=8)
    ax_h.set_title(era_titles[col], fontsize=10, fontweight='bold', pad=6)

    # ── Bottom row: map ──
    ax_m = axes[1, col]
    ax_m.set_facecolor(P['bg'])
    
    conus.plot(ax=ax_m, facecolor=P['land'], edgecolor=P['border'], linewidth=0.3)
    study_gdf = conus[conus['abbr'].isin(STATE_ORDER)]
    study_gdf.plot(ax=ax_m, facecolor=P['study'], edgecolor=P['border'], linewidth=0.3)
    
    if edges:
        for src, dest, w in sorted(edges, key=lambda e: e[2]):
            if src not in centroids or dest not in centroids:
                continue
            poly = build_arrow_polygon(
                centroids[src], centroids[dest], w, global_w_min, global_w_max)
            if poly is None:
                continue
            patch = MplPolygon(poly, closed=True, facecolor=P['arrow'],
                              edgecolor='none', alpha=1.0, zorder=3)
            ax_m.add_patch(patch)
    
    halo = [pe.withStroke(linewidth=2.0, foreground='white')]
    for s in STATE_ORDER:
        if s in centroids:
            x, y = centroids[s]
            ax_m.text(x, y, s, ha='center', va='center', fontsize=5.5,
                     fontweight='bold', color=P['label'], zorder=5,
                     fontfamily='Arial', path_effects=halo)
    
    ax_m.set_xlim(conus.total_bounds[0] - 80000, conus.total_bounds[2] + 80000)
    ax_m.set_ylim(conus.total_bounds[1] - 80000, conus.total_bounds[3] + 80000)
    ax_m.set_aspect('equal')
    ax_m.axis('off')

# Colorbar
cbar_ax = fig.add_axes([0.92, 0.55, 0.008, 0.30])
cbar = fig.colorbar(im, cax=cbar_ax)
cbar.set_label('Mean transitions', fontsize=10)
cbar.ax.tick_params(labelsize=9)

# Panel labels (lowercase)
for i, label in enumerate(['a', 'b', 'c', 'd']):
    axes[0, i].text(-0.05, 1.05, label, transform=axes[0, i].transAxes,
                    fontsize=12, fontweight='bold', va='top')
for i, label in enumerate(['e', 'f', 'g', 'h']):
    axes[1, i].text(0.0, 1.02, label, transform=axes[1, i].transAxes,
                    fontsize=12, fontweight='bold', va='top')

# fig.savefig(FIG_DIR / 'S_TransitionMaps_2x4.mean.png', dpi=300,
#             bbox_inches='tight', facecolor='white')
# fig.savefig(FIG_DIR / 'S_TransitionMaps_2x4.mean.pdf', dpi=300,
#             bbox_inches='tight', facecolor='white')
plt.show()
# print('Saved 2x4 figure.')

Arrow scaling: global w_min=1, w_max=12


/var/folders/ls/3swb_7515hv8jb0mjxshss480000gp/T/ipykernel_98066/305028867.py:118: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [8]:
# Difference between mean and median transition matrices
diff_matrices = {era: np.round(mean_matrices[era], 0) - median_matrices[era] for era in era_names}
diff_matrices = {era: np.round(diff_matrices[era], 0) for era in era_names}

[sum(sum(diff_matrices[era])) for era in era_names]

[np.float64(1.0), np.float64(1.0), np.float64(2.0), np.float64(0.0)]

In [9]:
# dataframe of each matrix 
matrices_df = pd.DataFrame({
    'era': [era for era in era_names for _ in range(n_states * n_states)],
    'origin': [STATE_ORDER[i] for era in era_names for i in range(n_states) for _ in range(n_states)],
    'destination': [STATE_ORDER[j] for era in era_names for _ in range(n_states) for j in range(n_states)],
    'mean_transitions': [(mean_matrices[era][i, j]) for era in era_names for i in range(n_states) for j in range(n_states)],
    'median_transitions': [median_matrices[era][i, j] for era in era_names for i in range(n_states) for j in range(n_states)],
    'diff_transitions': [diff_matrices[era][i, j] for era in era_names for i in range(n_states) for j in range(n_states)],
})
matrices_df[matrices_df['era'] == 'All'][['mean_transitions', 'median_transitions', 'diff_transitions']].sum()

mean_transitions      66.065766
median_transitions    63.000000
diff_transitions       1.000000
dtype: float64